In [1]:
%pip install -qq pyspark
import os
from pyspark.sql import SparkSession

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Set the target directory to '../data' relative to the notebook
data_dir = "../data"
os.makedirs(data_dir, exist_ok=True)

# Base URL for the 10BT Sample
base_url = "https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/main/sample/10BT"

# Loop through files 000 to 013
for i in range(14):
    filename = f"{i:03d}_00000.parquet"
    save_path = os.path.join(data_dir, filename)
    
    # Only download if missing
    if not os.path.exists(save_path):
        print(f"Downloading {filename}...")
        !wget -q -O {save_path} {base_url}/{filename}
    else:
        print(f"Skipping {filename} (Already exists)")

print("All 14 sample files downloaded.")

Skipping 000_00000.parquet (Already exists)
Skipping 001_00000.parquet (Already exists)
Skipping 002_00000.parquet (Already exists)
Skipping 003_00000.parquet (Already exists)
Skipping 004_00000.parquet (Already exists)
Skipping 005_00000.parquet (Already exists)
Skipping 006_00000.parquet (Already exists)
Skipping 007_00000.parquet (Already exists)
Skipping 008_00000.parquet (Already exists)
Skipping 009_00000.parquet (Already exists)
Skipping 010_00000.parquet (Already exists)
Skipping 011_00000.parquet (Already exists)
Skipping 012_00000.parquet (Already exists)
Skipping 013_00000.parquet (Already exists)
All 14 sample files downloaded.


In [3]:
# Create SparkSession with specified configurations
spark = SparkSession.builder \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "15g") \
    .config("spark.executor.instances", 7) \
    .getOrCreate()

spark

In [4]:
# Read all parquet files from the '../data' directory into a single DataFrame
df = spark.read.parquet("../data")

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import MinMaxScaler, VectorAssembler

# Filtering Invalid Documents

df = df.filter(
    (F.col("text").isNotNull()) &
    (F.length(F.col("text")) > 200) &
    (F.col("token_count") >= 50)
)

df.count()

9672072

In [6]:
# Feature Engineering using Spark SQL functions (regexp_extract, length, when, otherwise)

df = df.withColumn(
    "domain",
    F.regexp_extract(F.col("url"), r"https?://([^/]+)", 1)
)

df = df.withColumn(
    "length_bucket",
    F.when(F.col("token_count") < 500, "short")
     .when(F.col("token_count") < 2000, "medium")
     .otherwise("long")
)

In [7]:
# Scaling token_count, score, language_score using MinMaxScaler to the [0,1] range

assembler = VectorAssembler(
    inputCols=["token_count", "score", "language_score"],
    outputCol="numeric_features"
)

df = assembler.transform(df)

scaler = MinMaxScaler(
    inputCol="numeric_features",
    outputCol="scaled_features"
)

scaler_model = scaler.fit(df)
df = scaler_model.transform(df)

In [8]:
#  Apply dropDuplicates(["id"]) to remove duplicate documents

df = df.dropDuplicates(["id"])

In [9]:
# Defining the target variable (quality)

df = df.withColumn("label", F.col("int_score").cast(IntegerType()))

In [10]:
df.groupBy("int_score").count().orderBy("int_score").show()

+---------+-------+
|int_score|  count|
+---------+-------+
|        3|8383846|
|        4|1280796|
|        5|   7430|
+---------+-------+



In [ ]:
# Due to class imabalances (86.7% of documents are of quality bucket 3) we will use stratified sampling
# Keep all bucket 5
# Keep 30% of bucket 4
# Keep 5% of bucket 3